# Week 24, Day 1: Why we test

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w024d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def is_leap(year):
    return year % 4 == 0

print(is_leap(2024), is_leap(2023), is_leap(1900), is_leap(2000))

In [ ]:
def is_leap(year):
    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

def test_ordinary_leap_year():
    assert is_leap(2024)

def test_ordinary_year():
    assert not is_leap(2023)

def test_century_is_not_leap():
    assert not is_leap(1900)

def test_every_400_years_is_leap():
    assert is_leap(2000)

# A tiny test runner: call every test_ function and report.
for name, fn in list(globals().items()):
    if name.startswith("test_") and callable(fn):
        try:
            fn()
            print("PASS", name)
        except AssertionError:
            print("FAIL", name)

In [ ]:
def grade(mark):
    if mark >= 70:
        return "A"
    if mark >= 40:
        return "pass"
    return "fail"

def weak_tests(f):
    return f(90) == "A" and f(10) == "fail"

def strong_tests(f):
    return f(70) == "A" and f(69) == "pass" and f(40) == "pass" and f(39) == "fail" and f(90) == "A"

def broken(mark):              # off by one: > instead of >=
    if mark > 70:
        return "A"
    if mark > 40:
        return "pass"
    return "fail"

print(weak_tests(grade), weak_tests(broken))      # the weak tests can't tell them apart
print(strong_tests(grade), strong_tests(broken))

> **Key idea.** A test is a `test_` function that arranges an input, acts by calling the code, and asserts the result. Good tests check ordinary cases, boundaries and special cases, and are judged by the bugs they'd catch. Most tests should be fast unit tests. When you find a bug, write a failing test first, then fix it.

### Exercise: Test is_leap

Write tests for `is_leap(year)` from `calendar_rules.py`. Write at least three test functions. They must pass for the correct version and catch each of four broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('calendar_rules.py', 'w', encoding='utf-8') as f:
    f.write('def is_leap(year):\n    """True if year is a leap year in the Gregorian calendar."""\n    return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)\n')

In [ ]:
from calendar_rules import is_leap

def test_ordinary_leap_year():
    assert is_leap(2024)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"calendar_rules.py\").read()\nBROKEN = {\n    \"every fourth year\": \"def is_leap(y):\\n    return y % 4 == 0\\n\",\n    \"forgets 400\": \"def is_leap(y):\\n    return y % 4 == 0 and y % 100 != 0\\n\",\n    \"always true for centuries\": \"def is_leap(y):\\n    return y % 4 == 0 or y % 100 == 0\\n\",\n    \"wrong remainder\": \"def is_leap(y):\\n    return y % 4 == 1 or y % 400 == 0\\n\",\n}\ndef test_yours():\n    \"your tests pass on the correct code and catch every broken version\"\n    import ast\n    names = [n.name for n in ast.parse(source).body if isinstance(n, ast.FunctionDef) and n.name.startswith(\"test_\")]\n    assert len(names) >= 3, \"write at least three test functions\"\n    results = against(source, \"calendar_rules\", {\"correct\": GOOD, **BROKEN})\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [label for label in BROKEN if not results[label]]\n    write_file(\"calendar_rules.py\", GOOD)\n    assert not missed, f\"your tests don't catch these broken versions: {missed}\"")

### Exercise: Test the boundaries

`grade(mark)` in `marks.py` returns `"A"` for 70 and over, `"pass"` for 40 to 69, `"fail"` below 40, and raises `ValueError` for marks below 0 or above 100. Write tests that catch every broken version. To test that something raises an error, use this pattern:

```
def test_too_high():
    try:
        grade(101)
        assert False, "should have raised ValueError"
    except ValueError:
        pass
```

In [ ]:
# Run this cell first: it creates the files used below.
with open('marks.py', 'w', encoding='utf-8') as f:
    f.write('def grade(mark):\n    if not 0 <= mark <= 100:\n        raise ValueError("mark must be from 0 to 100")\n    if mark >= 70:\n        return "A"\n    if mark >= 40:\n        return "pass"\n    return "fail"\n')

In [ ]:
from marks import grade

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"marks.py\").read()\nBODY = \"\\n    if mark >= {a}:\\n        return 'A'\\n    if mark >= {p}:\\n        return 'pass'\\n    return 'fail'\\n\"\nBROKEN = {\n    \"A starts at 71\": \"def grade(mark):\\n    if not 0 <= mark <= 100:\\n        raise ValueError()\" + BODY.format(a=71, p=40),\n    \"pass starts at 41\": \"def grade(mark):\\n    if not 0 <= mark <= 100:\\n        raise ValueError()\" + BODY.format(a=70, p=41),\n    \"pass starts at 39\": \"def grade(mark):\\n    if not 0 <= mark <= 100:\\n        raise ValueError()\" + BODY.format(a=70, p=39),\n    \"no range check\": \"def grade(mark):\" + BODY.format(a=70, p=40),\n    \"100 rejected\": \"def grade(mark):\\n    if not 0 <= mark < 100:\\n        raise ValueError()\" + BODY.format(a=70, p=40),\n}\ndef test_yours():\n    \"your tests catch every broken version\"\n    results = against(source, \"marks\", {\"correct\": GOOD, **BROKEN})\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [label for label in BROKEN if not results[label]]\n    write_file(\"marks.py\", GOOD)\n    assert not missed, f\"your tests don't catch these broken versions: {missed}\"")

### Exercise: Test a list function

`second_largest(items)` in `ranking.py` returns the second largest **different** value, or `None` if there isn't one. Write tests that catch every broken version. Think about repeats, short lists and negative numbers.

In [ ]:
# Run this cell first: it creates the files used below.
with open('ranking.py', 'w', encoding='utf-8') as f:
    f.write('def second_largest(items):\n    best = second = None\n    for x in items:\n        if best is None or x > best:\n            if best is not None:\n                second = best\n            best = x\n        elif x != best and (second is None or x > second):\n            second = x\n    return second\n')

In [ ]:
from ranking import second_largest

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"ranking.py\").read()\nBROKEN = {\n    \"sorts and takes [-2]\": \"def second_largest(items):\\n    return sorted(items)[-2] if len(items) > 1 else None\\n\",\n    \"ignores later seconds\": \"def second_largest(items):\\n    best = second = None\\n    for x in items:\\n        if best is None or x > best:\\n            second, best = best, x\\n    return second\\n\",\n    \"starts at zero\": \"def second_largest(items):\\n    best = second = 0\\n    for x in items:\\n        if x > best:\\n            second, best = best, x\\n        elif best > x > second:\\n            second = x\\n    return second if len(set(items)) > 1 else None\\n\",\n    \"crashes on empty\": \"def second_largest(items):\\n    s = sorted(set(items))\\n    return s[-2] if len(s) != 1 else None\\n\",\n}\ndef test_yours():\n    \"your tests catch every broken version\"\n    results = against(source, \"ranking\", {\"correct\": GOOD, **BROKEN})\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [label for label in BROKEN if not results[label]]\n    write_file(\"ranking.py\", GOOD)\n    assert not missed, f\"your tests don't catch these broken versions: {missed}\"")

### Exercise: A bug, a test, a fix

Users report that `word_count("  hello   world ")` returns 7 instead of 2. Following the three steps: first add a regression test `test_extra_spaces` (and one more test of your own) to the **test code** in the editor, then fix `word_count` in the same editor so all your tests pass. The checks run your tests against your fixed function and against the original buggy one (which they must catch).

In [ ]:
def word_count(text):
    return len(text.split(" "))

def test_simple():
    assert word_count("one two three") == 3

In [ ]:
# Check: run the exercise cell above first, then this one.
check("BUGGY = 'def word_count(text):\\n    return len(text.split(\" \"))\\n'\ndef test_fixed():\n    \"your word_count is fixed\"\n    assert word_count(\"  hello   world \") == 2 and word_count(\"\") == 0 and word_count(\"one\") == 1\ndef test_regression():\n    \"your tests include test_extra_spaces and catch the original bug\"\n    import ast\n    tree = ast.parse(source)\n    names = [n.name for n in tree.body if isinstance(n, ast.FunctionDef) and n.name.startswith(\"test_\")]\n    assert \"test_extra_spaces\" in names, \"add a test called test_extra_spaces\"\n    tests_only = \"from counter import word_count\\n\" + \"\\n\".join(\n        ast.unparse(n) for n in tree.body if isinstance(n, ast.FunctionDef) and n.name.startswith(\"test_\"))\n    fixed = \"\\n\".join(ast.unparse(n) for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == \"word_count\")\n    results = against(tests_only, \"counter\", {\"fixed\": fixed, \"buggy\": BUGGY})\n    assert results[\"fixed\"] == [], f\"these fail on your fixed version: {results['fixed']}\"\n    assert results[\"buggy\"], \"your tests don't catch the original bug\"")

### Exercise: Which kind of test?

Label each test as `"unit"`, `"integration"` or `"end-to-end"`.

In [ ]:
a = ___   # checks that parse_date("31/10/2026") returns date(2026, 10, 31)
b = ___   # runs `python3 report.py data.csv` and checks the report file it writes
c = ___   # reads a small CSV with the csv module, cleans it with clean(), and checks the JSON produced
d = ___   # checks that is_leap(1900) is False

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"unit\" and b == \"end-to-end\" and c == \"integration\" and d == \"unit\"")

### Bug hunt: A test that tests nothing

This test always passes, even if `double` is completely wrong, because it never asserts anything: it just computes a comparison and throws the result away. Fix it so it checks the result. (The checks run it against a broken `double`.)

In [ ]:
def double(x):
    return x * 2

def test_double():
    double(3) == 6
    double(0) == 0

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_catches():\n    \"the test now checks something\"\n    import ast\n    tests_only = \"from doubler import double\\n\" + \"\\n\".join(\n        ast.unparse(n) for n in ast.parse(source).body if isinstance(n, ast.FunctionDef) and n.name.startswith(\"test_\"))\n    r = against(tests_only, \"doubler\", {\"good\": \"def double(x):\\n    return x * 2\\n\", \"bad\": \"def double(x):\\n    return x + 2\\n\"})\n    assert r[\"good\"] == [] and r[\"bad\"], \"the test still doesn't catch a wrong double\"")

### Bug hunt: The test is wrong, not the code

`average([1, 2])` is 1.5, and the function is correct, but this test fails. A failing test doesn't always mean the code is wrong: sometimes the test is. Fix the test.

In [ ]:
def average(xs):
    return sum(xs) / len(xs)

def test_average():
    assert average([1, 2]) == 1
    assert average([4, 4, 4]) == 4

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_now_passes():\n    \"the test passes on the correct code and still catches a bug\"\n    import ast\n    tests_only = \"from avg import average\\n\" + \"\\n\".join(\n        ast.unparse(n) for n in ast.parse(source).body if isinstance(n, ast.FunctionDef) and n.name.startswith(\"test_\"))\n    r = against(tests_only, \"avg\", {\"good\": \"def average(xs):\\n    return sum(xs) / len(xs)\\n\", \"bad\": \"def average(xs):\\n    return sum(xs) // len(xs)\\n\"})\n    assert r[\"good\"] == [], \"the test still fails on the correct code\"\n    assert r[\"bad\"], \"the test should catch an average that rounds down\"")

### Puzzle: Arrange, act, assert

Arrange the lines into a test with the three parts in order. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Answer from what you know.

In [ ]:
a = ___   # True or False: a test that passes proves the code has no bugs
b = ___   # the three parts of a test, as a list of three lower-case words
c = ___   # what a test written for a fixed bug is called (one word)
d = ___   # which kind of test should a project have most of?

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a is False and b == [\"arrange\", \"act\", \"assert\"] and c == \"regression\" and d == \"unit\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: test a password checker

`strength(password)` in `passwords.py` returns `"weak"`, `"medium"` or `"strong"` by these rules: under 8 characters is always weak; otherwise count how many of these it has: a lower-case letter, a capital letter, a digit, a character that's none of those. 4 kinds and at least 12 characters is strong; 3 or more kinds is medium; otherwise weak. Write a thorough set of tests that passes on the correct code and catches all **six** broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('passwords.py', 'w', encoding='utf-8') as f:
    f.write('def strength(password):\n    if len(password) < 8:\n        return "weak"\n    kinds = sum([\n        any(c.islower() for c in password),\n        any(c.isupper() for c in password),\n        any(c.isdigit() for c in password),\n        any(not c.isalnum() for c in password),\n    ])\n    if kinds == 4 and len(password) >= 12:\n        return "strong"\n    if kinds >= 3:\n        return "medium"\n    return "weak"\n')

In [ ]:
from passwords import strength

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"passwords.py\").read()\nBROKEN = {\n    \"length 7 allowed\": GOOD.replace(\"len(password) < 8\", \"len(password) < 7\"),\n    \"strong at 11\": GOOD.replace(\"len(password) >= 12\", \"len(password) >= 11\"),\n    \"symbols not counted\": GOOD.replace(\"any(not c.isalnum() for c in password),\", \"False,\"),\n    \"capitals counted as lower\": GOOD.replace(\"any(c.isupper() for c in password)\", \"any(c.islower() for c in password)\"),\n    \"medium needs 4\": GOOD.replace(\"kinds >= 3\", \"kinds >= 4\"),\n    \"strong ignores length\": GOOD.replace('kinds == 4 and len(password) >= 12', 'kinds == 4'),\n}\ndef test_yours():\n    \"your tests catch all six broken versions\"\n    results = against(source, \"passwords\", {\"correct\": GOOD, **BROKEN})\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [label for label in BROKEN if not results[label]]\n    write_file(\"passwords.py\", GOOD)\n    assert not missed, f\"your tests don't catch these broken versions: {missed}\"")

## Warm-up

It looks right for 2024 and 2023, and someone trying it by hand might stop there. But 1900 was **not** a leap year: century years are only leap years if they divide by 400. A test that checks 1900 catches the bug at once, and keeps catching it if anyone ever breaks the function again.

## Testing by hand, and by code

So far you've tested your code by running it and looking at the output. That works for one small program, once. It stops working when:

- the program is large, and checking everything by hand takes an hour;
- you change something, and need to recheck everything the change might have broken;
- someone else changes your code, and doesn't know what it's supposed to do;
- the bug only shows up for an input you didn't think to try today.

An **automated test** is code that runs your code with chosen inputs and checks the results. Once written, it runs in a fraction of a second, as often as you like, and never gets bored or forgets a case. Professional projects have thousands of them, run after every change.

Tests give you:

- **Confidence to change code.** Rename, tidy, speed up: if the tests still pass, the behaviour is still right. This is what makes improving code safe.
- **Bugs found early**, when they're cheap: a minute after you wrote them, not a month later.
- **Documentation that can't go out of date.** A test shows exactly what a function should do with each kind of input.
- **Better design.** Code that's hard to test (that prints instead of returning, or depends on the date) is usually hard to use as well.

## Your first tests

A test is a function whose name starts with `test_`. It usually has three parts, often called **arrange, act, assert**:

1. **Arrange**: set up the input.
2. **Act**: call the code being tested.
3. **Assert**: check the result with `assert`, which does nothing if its condition is true and raises `AssertionError` if it's false.

Each test checks **one** behaviour, and its name says which. When one fails, the name alone tells you what broke.

An `assert` can carry a message, shown when it fails, and it's worth adding when the plain comparison isn't obvious:

```
assert total == 31.5, f"expected 31.5, got {total}"
```

Tomorrow you'll use **pytest**, which finds and runs `test_` functions automatically and explains failures in detail, so you won't write runners like the loop above. The tests themselves look exactly the same.

## Tests that catch bugs

What makes a test **good** is the bugs it would catch. A test that only checks easy inputs passes for plenty of wrong code. Good tests check:

- **ordinary cases**: the obvious inputs;
- **boundaries**: where behaviour changes. Ages 17 and 18 for "adult"; marks of 39, 40 and 41 for a pass at 40; the first and last items of a list;
- **special cases**: empty lists, zero, negative numbers, a single item, repeats;
- **the cases you're unsure about**, which is where bugs hide.

One way to judge a set of tests: imagine (or write) small **wrong** versions of the code, and ask whether the tests notice. Each wrong version a test catches is a bug that can't sneak in later. Professional testers call this **mutation testing**, and the exercises this week use it to check your tests: they run your tests against the correct code (every test must pass) and against several broken versions (your tests must fail on each).

The weak tests pass for both versions; only tests at the boundaries tell them apart.

## Kinds of test

| Kind | Checks | Speed | Example |
|---|---|---|---|
| **Unit** test | one small piece (a function) on its own | milliseconds | `is_leap(1900)` is False |
| **Integration** test | several pieces working together | slower | reading a CSV, cleaning it and writing JSON |
| **End-to-end** test | the whole program, as a user would | slowest | run the script with arguments, check the output file |

Most tests should be unit tests (fast and precise about what broke), with fewer integration tests and only a few end-to-end ones. This shape is often drawn as the "test pyramid".

A **regression test** is one written for a bug that was found and fixed, to make sure it never comes back ("regresses"). Every bug you fix deserves one.

## A bug, a test, a fix

When you find a bug, follow these steps in order:

1. **Write a test that fails because of the bug.** This proves you understand it, and that the test can see it.
2. **Fix the code.**
3. **Run the test again.** It passes now; and so do all the others, so the fix didn't break anything else.

The test then stays in the project for good, as a regression test.

## Practice

In these exercises, write **test functions** with `assert`. Each exercise has the code under test in a file you import. The checks run your tests against the correct code (all must pass) and against several broken versions (your tests must catch each one).

## Quiz

## Daily challenge

## Before you go

Pick a function you wrote in an earlier week and write three tests for it: an ordinary case, a boundary, and a special case. Then deliberately break the function in a small way and see whether your tests notice. Tomorrow: pytest, the tool that runs tests for nearly every Python project.